# 1. **Preprocessing**

In [ ]:
import pandas as pd
import re

df = pd.read_csv('/content/dataset_mentah_twitter.csv')
tweets_column = 'full_text'

print(f"Total Data: {len(df)}")

df = df.dropna(subset=[tweets_column])
df = df[df[tweets_column].str.strip() != '']



In [ ]:
# Standarization
def standardize_indobertweet_cased(text):
    text = str(text)
    text = re.sub(r'@\w+', '@USER', text) # Tag
    text = re.sub(r'http\S+|www\.\S+', 'HTTPURL', text) # Link
    text = re.sub(r'[^\w\s\$\.,!?\-@#]', '', text) # Symbol
    text = re.sub(r'\s+', ' ', text).strip() # Redundant white space
    return text

df['clean_text'] = df[tweets_column].apply(standardize_indobertweet_cased)

In [ ]:


# Out of context and promotional filter
blacklist = [
    r'\blistrik\b', r'\bpln\b', r'\bpulsa\b', r'\bmeteran\b', r'\bshopee\b', r'\btokopedia\b',
    r'\btokped\b', r'\bline\b', r'\btiktok\b', r'\broblox\b', r'\blazada\b', r'\bgopay\b',
    r'\bmlbb\b', r'mobile legend', r'\bwebtoon\b', r'\bhayday\b', r'\bovo\b', r'\bsupercell\b',
    r'\bmakeup\b', r'\bseventeen\b', r'\bnana\b', r'\bpreloved\b', r'\bbarenbliss\b',
    r'\bparfum\b', r'\bwts\b', r'\bwtb\b', r'flash\s?sale', r'cashback',
    r'\bgiveaway\b', r'rt\s?(&|dan)?\s?follow', r'link (in|di) bio', r'\bjoin grup\b'
]
blacklist_join = '|'.join(blacklist)

def filter_blacklist(text):
    if re.search(blacklist_join, text, flags=re.IGNORECASE):
        return True
    return False

df = df[~df['clean_text'].apply(filter_blacklist)]

In [ ]:
# Hybrid Language Filtering
indo_func_words = ['dari', 'yang', 'yg', 'dan', 'ini', 'itu', 'ada', 'udah', 'saja', 'aja', 'buat', 'kalo', 'kalau', 'sama', 'kok', 'sih', 'dong']
idfw_join = r'\b(' + '|'.join(indo_func_words) + r')\b'

def has_idfw(text):
    return bool(re.search(idfw_join, text, flags=re.IGNORECASE))

lang_column = 'lang'

mask_is_id = df[lang_column] == 'id'
mask_has_idfw = df['clean_text'].apply(has_idfw)

df = df[mask_is_id | mask_has_idfw]

In [ ]:
# Spam Filtering
df = df[df['clean_text'].str.len() >= 15]
df = df[df['clean_text'].str.count('@USER') < 5]
df = df[df['clean_text'].str.count('#') < 6]

In [ ]:
print(f"Total Data After Cleaned: {len(df)}")
df.to_csv('/content/dataset_preprocessed.csv', index=False)

# 2. **Splitting (Gold Silver)**

## a. Load LKB

In [ ]:
import pandas as pd
import re
import json

df = pd.read_csv('/content/dataset_preprocessed.csv')
tweets_column = 'clean_text'

def load_lkb(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    crypto_assets = set()

    for key, info in data.items():
        crypto_assets.add(key.lower().strip())

        if isinstance(info, dict):
            ticker = info.get("symbol", "")
            if ticker:
                crypto_assets.add(ticker.lower().strip())

            official_name = info.get("official_name", "")
            if official_name:
                crypto_assets.add(official_name.lower().strip())
                cleaned_name = re.sub(r'\s*\(.*?\)\s*', '', official_name).strip()
                if cleaned_name:
                    crypto_assets.add(cleaned_name.lower())

            for alias in info.get("aliases", []):
                if alias:
                    crypto_assets.add(alias.lower().strip())

    if "" in crypto_assets:
        crypto_assets.remove("")

    return crypto_assets

lkb_unambigous = load_lkb('/content/lkb_cleaned.json')
lkb_ambigous = load_lkb('/content/lkb_case_sensitive.json')

## b. Splitting

In [ ]:
blue_chips = {'btc', 'bitcoin', 'eth', 'ethereum', 'sol', 'solana', 'bnb', 'xrp', 'ripple'}
all_crypto = lkb_unambigous.union(lkb_ambigous)
altcoins = all_crypto - blue_chips

# Gold Standard
def stratified_sampling(text):
    text_lower = str(text).lower()
    tokens = set(re.findall(r'\b\w+\b', text_lower))

    if not tokens.isdisjoint(blue_chips):
        return 'Strata_A_BlueChip'
    elif not tokens.isdisjoint(altcoins):
        return 'Strata_B_Altcoin'
    elif any(kw in text_lower for kw in financial_keywords):
        return 'Strata_C_Background'
    else:
        return 'Noise'

df['Strata_Temp'] = df[tweets_column].apply(stratified_sampling)

sampel_a = df[df['Strata_Temp'] == 'Strata_A_BlueChip'].sample(n=450, random_state=42)
sampel_b = df[df['Strata_Temp'] == 'Strata_B_Altcoin'].sample(n=750, random_state=42)
sampel_c = df[df['Strata_Temp'] == 'Strata_C_Background'].sample(n=300, random_state=42)

df_gold = pd.concat([sampel_a, sampel_b, sampel_c]).sample(frac=1, random_state=42)

# Silver Standard
df_silver = df.drop(df_gold.index)


df_gold = df_gold.drop(columns=['Strata_Temp']).reset_index(drop=True)
df_silver = df_silver.drop(columns=['Strata_Temp']).reset_index(drop=True)

df_gold.to_csv('/content/gold_standard_dataset.csv', index=False)
df_silver.to_csv('/content/silver_standard_dataset.csv', index=False)


print(f"Total Cleaned Dataset     : {len(df)}")
print(f"Gold Standard Dataset     : {len(df_gold)}")
print(f"Silver Standard Dataset   : {len(df_silver)}")

# 3. **Labeling**

## a. Load LKB

In [ ]:
import pandas as pd
import re
import json

df = pd.read_csv('/content/dataset_preprocessed.csv')
tweets_column = 'clean_text'

def load_lkb(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    crypto_assets = set()

    for key, info in data.items():
        crypto_assets.add(key.lower().strip())

        if isinstance(info, dict):
            ticker = info.get("symbol", "")
            if ticker:
                crypto_assets.add(ticker.lower().strip())

            official_name = info.get("official_name", "")
            if official_name:
                crypto_assets.add(official_name.lower().strip())
                cleaned_name = re.sub(r'\s*\(.*?\)\s*', '', official_name).strip()
                if cleaned_name:
                    crypto_assets.add(cleaned_name.lower())

            for alias in info.get("aliases", []):
                if alias:
                    crypto_assets.add(alias.lower().strip())

    if "" in crypto_assets:
        crypto_assets.remove("")

    return crypto_assets

lkb_unambigous = load_lkb('/content/lkb_cleaned.json')
lkb_ambigous = load_lkb('/content/lkb_case_sensitive.json')

## b. Distant Labelling Function

In [ ]:
financial_keywords = [
    'serok', 'cuan', 'nyangkut', 'terbang', 'nyungsep', 'rungkad', 'haka', 'cicil',
    'pump', 'dump', 'hold', 'hodl', 'longsor', 'moon', 'whale', 'bullish', 'bearish',
    'kripto', 'crypto', 'koin', 'coin', 'token', 'market',
    'pasar', 'naik', 'turun', 'profit', 'loss', 'akumulasi', 'wallet', 'dompet', 'swap'
]

def distant_labelling(text, lkb_unambigous, lkb_ambigous, financial_keywords):
    tokens = re.findall(r'@USER|HTTPURL|\#\w+|\$\w+|[a-zA-Z0-9_]+|[^\w\s]', str(text))
    labels = ['O'] * len(tokens)

    i = 0
    while i < len(tokens):
        ori_token = tokens[i]
        token_core = re.sub(r'^[\$\#]', '', ori_token)
        token_core_lower = token_core.lower()

        if (ori_token.startswith('$') or ori_token.startswith('#')) and len(ori_token) > 1:
            is_price = re.match(r'^[\d\.,]+[kKmMbB]?$', token_core)
            if not is_price:
                if token_core_lower in lkb_unambigous or token_core_lower in lkb_ambigous:
                    labels[i] = 'B-CRYPTO'
                i += 1
                continue

        matched = False

        for window in range(4, 0, -1):
            if i + window <= len(tokens):
                phrase_tokens = tokens[i:i+window]
                phrase_lower = ' '.join(phrase_tokens).lower()

                # Unambigous
                if phrase_lower in lkb_unambigous:
                    labels[i] = 'B-CRYPTO'
                    for j in range(1, window):
                        labels[i+j] = 'I-CRYPTO'
                    i += window
                    matched = True
                    break

                # Ambigous tokens
                if window == 1 and phrase_lower in lkb_ambigous:
                    if ori_token.isupper():
                        left_bound = max(0, i - 4)
                        right_bound = min(len(tokens), i + 5)
                        context_words = [t.lower() for t in tokens[left_bound:right_bound]]

                        if any(kw in context_words for kw in financial_keywords):
                            labels[i] = 'B-CRYPTO'
                            i += 1
                            matched = True
                            break

        if not matched:
            i += 1

    return list(zip(tokens, labels))

## c. Silver Standard BIO

In [ ]:
df_silver_raw = pd.read_csv('/content/silver_standard_dataset.csv')

silver_row = []
for idx, row in df_silver_raw.iterrows():
    teks = row['clean_text']
    if pd.isna(teks): continue

    hasil_bio = distant_labelling(teks, lkb_umum, lkb_ambigu, konteks_finansial)

    for token, label in hasil_bio:
        silver_row.append({
            'Sentence_ID': f"Silver_{idx+1}",
            'Token': token,
            'Label': label
        })

df_silver_bio = pd.DataFrame(silver_row)
df_silver_bio.to_csv('/content/silver_standard_bio.csv', index=False)

## d. Gold Standard BIO Template

In [ ]:
df_gold_raw = pd.read_csv('/content/gold_standard_dataset.csv')

gold_row = []
for idx, row in df_gold_raw.iterrows():
    teks = str(row['clean_text'])
    tweet_url = str(row.get('tweet_url', ''))

    tokens = re.findall(r'@USER|HTTPURL|\#\w+|\$\w+|[a-zA-Z0-9_]+|[^\w\s]', teks)

    for i, token in enumerate(tokens):
        tampilkan_url = tweet_url if i == 0 else ""

        gold_row.append({
            'Sentence_ID': f"Gold_{idx+1}",
            'Token': token,
            'Label': '', # For Manual Labelling
            'Referensi_URL': tampilkan_url
        })

    gold_row.append({'Sentence_ID': '', 'Token': '', 'Label': '', 'Referensi_URL': ''})

df_gold_bio = pd.DataFrame(baris_gold)
df_gold_bio.to_csv('/content/gold_standard_bio.csv', index=False)

# 4. **Model Training**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## a. Base Model

In [ ]:
!pip install transformers datasets evaluate seqeval pytorch-crf

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback
from datasets import Dataset
from torchcrf import CRF
import numpy as np
import evaluate

class IndoBERTweetCRF(nn.Module):
    def __init__(self, model_checkpoint, num_labels):
        super(IndoBERTweetCRF, self).__init__()

        self.config = AutoConfig.from_pretrained(model_checkpoint)
        self.bert = AutoModel.from_pretrained(model_checkpoint, config=self.config)

        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)

        emissions = self.classifier(sequence_output)

        if labels is not None:
            crf_mask = attention_mask.type(torch.uint8)
            safe_labels = torch.where(labels == -100, torch.tensor(0).to(labels.device), labels)

            loss = -self.crf(emissions, safe_labels, mask=crf_mask, reduction='mean')
            preds = self.crf.decode(emissions, mask=crf_mask)

            max_seq_len = emissions.shape[1]
            padded_preds = []
            for p in preds:
                padded_preds.append(p + [0] * (max_seq_len - len(p)))

            logits = torch.tensor(padded_preds).to(emissions.device)

            return {"loss": loss, "logits": logits}
        else:
            crf_mask = attention_mask.type(torch.uint8)
            preds = self.crf.decode(emissions, mask=crf_mask)
            return {"logits": preds}


label_list = ["O", "B-CRYPTO", "I-CRYPTO"]
label_to_id = {l: i for i, l in enumerate(label_list)}
num_labels = len(label_list)


model_checkpoint = "indolem/indobertweet-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = IndoBERTweetCRF(model_checkpoint, num_labels)
model.to("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
df_silver = pd.read_csv('/content/silver_standard_bio.csv')
df_silver = df_silver.dropna(subset=['Token', 'Label'])

sentences = []
labels = []
for seq_id, group in df_silver.groupby('Sentence_ID', sort=False):
    sentences.append(group['Token'].astype(str).tolist())
    labels.append(group['Label'].astype(str).tolist())

def tokenize_and_align_labels(texts, tags):
    tokenized_inputs = tokenizer(
        texts,
        is_split_into_words=True,
        truncation=True,
        max_length=128,
        padding="max_length"
    )

    aligned_labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                clean_label = label[word_idx] if label[word_idx] in label_to_id else "O"
                label_ids.append(label_to_id[clean_label])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

tokenized_datasets = tokenize_and_align_labels(sentences, labels)

silver_dataset_full = Dataset.from_dict({
    'input_ids': tokenized_datasets['input_ids'],
    'attention_mask': tokenized_datasets['attention_mask'],
    'labels': tokenized_datasets['labels']
})

# 90:10 Train Eval
split_dataset = silver_dataset_full.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"Splitted Silver")
print(f"Train Set : {len(train_dataset)} Sentences")
print(f"Eval Set  : {len(eval_dataset)} Sentences")


silver_dataset = Dataset.from_dict({
    'input_ids': tokenized_datasets['input_ids'],
    'attention_mask': tokenized_datasets['attention_mask'],
    'labels': tokenized_datasets['labels']
})

In [ ]:
class CRFTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/indobertweet-crf-base",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    logging_steps=50,
    fp16=True if torch.cuda.is_available() else False
)

trainer = CRFTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # 2 Patience early stopping
)

In [ ]:
trainer.train()

In [ ]:
model_final_path_drive = "/content/drive/MyDrive/indobertweet-crf-base-final"
model_final_path_colab = "/content/indobertweet-crf-base-final"

trainer.save_model(model_final_path_drive)
trainer.save_model(model_final_path_colab)

tokenizer.save_pretrained(model_final_path_drive)
tokenizer.save_pretrained(model_final_path_colab)

### Base Model Evaluation

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import evaluate
from transformers import AutoModel, AutoConfig, AutoTokenizer
from torchcrf import CRF
from safetensors.torch import load_file

seqeval = evaluate.load("seqeval")
df_gold = pd.read_csv('/content/gold_standard_bio_labeled.csv') # Using gold standard to evaluate base model
df_gold = df_gold.dropna(subset=['Token', 'Label'])

sentences_test = []
labels_test = []
for seq_id, group in df_gold.groupby('Sentence_ID', sort=False):
    sentences_test.append(group['Token'].astype(str).tolist())
    labels_test.append(group['Label'].astype(str).tolist())

print(f"Eval dataset: {len(sentences_test)} Sentences.")


In [ ]:
class IndoBERTweetCRF(nn.Module):
    def __init__(self, model_checkpoint, num_labels):
        super(IndoBERTweetCRF, self).__init__()
        self.config = AutoConfig.from_pretrained(model_checkpoint)
        self.bert = AutoModel.from_pretrained(model_checkpoint, config=self.config)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)
        emissions = self.classifier(sequence_output)

        if labels is not None:
            pass
        else:
            crf_mask = attention_mask.type(torch.uint8)
            preds = self.crf.decode(emissions, mask=crf_mask)
            return {"logits": preds}

label_list = ["O", "B-CRYPTO", "I-CRYPTO"]

In [ ]:
base_model_path = "/content/drive/MyDrive/indobertweet-crf-base-final"

if not os.path.exists(base_model_path):
    raise FileNotFoundError(f"{base_model_path} not found.")

tokenizer = AutoTokenizer.from_pretrained(base_model_path)
model = IndoBERTweetCRF("indolem/indobertweet-base-uncased", len(label_list))

file_safetensors = os.path.join(base_model_path, "model.safetensors")
file_bin = os.path.join(base_model_path, "pytorch_model.bin")

if os.path.exists(file_safetensors):
    state_dict = load_file(file_safetensors)
elif os.path.exists(file_bin):
    state_dict = torch.load(file_bin, map_location=torch.device('cpu'))
else:
    raise FileNotFoundError("Base model weight not found.")

model.load_state_dict(state_dict)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

In [ ]:
true_predictions = []
true_labels = []

for i in range(len(sentences_test)):
    words = sentences_test[i]
    tags = labels_test[i]

    inputs_hf = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
    word_ids = inputs_hf.word_ids(batch_index=0)
    inputs = {k: v.to(device) for k, v in inputs_hf.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        preds = outputs["logits"][0]

    pred_label_words = []
    true_label_words = []
    previous_word_idx = None

    for j, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            if j < len(preds):
                pred_label_words.append(label_list[preds[j]])
                true_label_words.append(tags[word_idx])

        previous_word_idx = word_idx

    true_predictions.append(pred_label_words)
    true_labels.append(true_label_words)

In [ ]:
results = seqeval.compute(predictions=true_predictions, references=true_labels)

print('Base model evaluation on Gold Standard')
print(f"Precision : {results['overall_precision']:.4f}")
print(f"Recall    : {results['overall_recall']:.4f}")
print(f"F1-Score  : {results['overall_f1']:.4f}")

## b. Hybrid Fine-Tuning (5-Fold Cross Validation)

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import evaluate
from transformers import AutoModel, AutoConfig, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback
from datasets import Dataset
from torchcrf import CRF
from safetensors.torch import load_file
from sklearn.model_selection import KFold

class IndoBERTweetCRF(nn.Module):
    def __init__(self, model_checkpoint, num_labels):
        super(IndoBERTweetCRF, self).__init__()
        self.config = AutoConfig.from_pretrained(model_checkpoint)
        self.bert = AutoModel.from_pretrained(model_checkpoint, config=self.config)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)
        emissions = self.classifier(sequence_output)

        if labels is not None:
            crf_mask = attention_mask.type(torch.uint8)
            safe_labels = torch.where(labels == -100, torch.tensor(0).to(labels.device), labels)
            loss = -self.crf(emissions, safe_labels, mask=crf_mask, reduction='mean')
            preds = self.crf.decode(emissions, mask=crf_mask)
            max_seq_len = emissions.shape[1]
            padded_preds = [p + [0] * (max_seq_len - len(p)) for p in preds]
            logits = torch.tensor(padded_preds).to(emissions.device)
            return {"loss": loss, "logits": logits}
        else:
            crf_mask = attention_mask.type(torch.uint8)
            preds = self.crf.decode(emissions, mask=crf_mask)
            return {"logits": preds}

    model.eval()
    true_predictions = []
    true_labels = []

    for i in range(len(X_test)):
        words = X_test[i]
        tags = y_test[i]

        inputs_hf = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
        word_ids = inputs_hf.word_ids(batch_index=0)
        inputs = {k: v.to(device) for k, v in inputs_hf.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            preds = outputs["logits"][0]

        pred_label_words = []
        true_label_words = []
        previous_word_idx = None
        for j, word_idx in enumerate(word_ids):
            if word_idx is None: continue
            elif word_idx != previous_word_idx:
                if j < len(preds):
                    pred_label_words.append(label_list[preds[j]])
                    true_label_words.append(tags[word_idx])
            previous_word_idx = word_idx

        true_predictions.append(pred_label_words)
        true_labels.append(true_label_words)

seqeval = evaluate.load("seqeval")
label_list = ["O", "B-CRYPTO", "I-CRYPTO"]
label_to_id = {l: i for i, l in enumerate(label_list)}
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df_gold = pd.read_csv('/content/gold_standard_bio_labeled.csv')
df_gold = df_gold.dropna(subset=['Token', 'Label'])

sentences_all = []
labels_all = []
for seq_id, group in df_gold.groupby('Sentence_ID', sort=False):
    sentences_all.append(group['Token'].astype(str).tolist())
    labels_all.append(group['Label'].astype(str).tolist())

sentences_all = np.array(sentences_all, dtype=object)
labels_all = np.array(labels_all, dtype=object)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []
base_model_path = "/content/indobertweet-crf-base-final"
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

def tokenize_and_align_labels(texts, tags):
    tokenized_inputs = tokenizer(texts.tolist(), is_split_into_words=True, truncation=True, max_length=128, padding="max_length")
    aligned_labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                clean_label = label[word_idx] if label[word_idx] in label_to_id else "O"
                label_ids.append(label_to_id[clean_label])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)
    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

for fold, (train_idx, test_idx) in enumerate(kf.split(sentences_all)):
    print(f"Fold: {fold + 1}/5")

    X_train, X_test = sentences_all[train_idx], sentences_all[test_idx]
    y_train, y_test = labels_all[train_idx], labels_all[test_idx]

    train_encodings = tokenize_and_align_labels(X_train, y_train)
    eval_encodings = tokenize_and_align_labels(X_test, y_test)

    train_dataset = Dataset.from_dict({'input_ids': train_encodings['input_ids'], 'attention_mask': train_encodings['attention_mask'], 'labels': train_encodings['labels']})
    eval_dataset = Dataset.from_dict({'input_ids': eval_encodings['input_ids'], 'attention_mask': eval_encodings['attention_mask'], 'labels': eval_encodings['labels']})

    model = IndoBERTweetCRF("indolem/indobertweet-base-uncased", len(label_list))
    file_safetensors = os.path.join(base_model_path, "model.safetensors")
    file_bin = os.path.join(base_model_path, "pytorch_model.bin")
    state_dict = load_file(file_safetensors) if os.path.exists(file_safetensors) else torch.load(file_bin, map_location=torch.device('cpu'))
    model.load_state_dict(state_dict)
    model.to(device)

    class CRFTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            outputs = model(**inputs)
            loss = outputs["loss"]
            return (loss, outputs) if return_outputs else loss

    training_args = TrainingArguments(
        output_dir=f"/content/drive/MyDrive/fold_{fold}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="loss",
        greater_is_better=False,
        logging_steps=50,
        fp16=True if torch.cuda.is_available() else False
    )

    trainer = CRFTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    model.eval()
    true_predictions = []
    true_labels = []

    for i in range(len(X_test)):
        words = X_test[i]
        tags = y_test[i]

        inputs_hf = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
        word_ids = inputs_hf.word_ids(batch_index=0)
        inputs = {k: v.to(device) for k, v in inputs_hf.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            preds = outputs["logits"][0]

        pred_label_words = []
        true_label_words = []
        previous_word_idx = None
        for j, word_idx in enumerate(word_ids):
            if word_idx is None: continue
            elif word_idx != previous_word_idx:
                if j < len(preds):
                    pred_label_words.append(label_list[preds[j]])
                    true_label_words.append(tags[word_idx])
            previous_word_idx = word_idx

        true_predictions.append(pred_label_words)
        true_labels.append(true_label_words)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    f1 = results['overall_f1']
    precision = results['overall_precision']
    recall = results['overall_recall']

    print(f"Result (Fold-{fold + 1})  | P: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f}\n")
    fold_results.append((precision, recall, f1))

print('\n5-Fold Cross Evaluaition Results:')
precisions = [x[0] for x in fold_results]
recalls = [x[1] for x in fold_results]
f1_scores = [x[2] for x in fold_results]

print(f"Mean Precision : {np.mean(precisions):.4f} (± {np.std(precisions):.4f})")
print(f"Mean Recall    : {np.mean(recalls):.4f} (± {np.std(recalls):.4f})")
print(f"Mean F1-Score  : {np.mean(f1_scores):.4f} (± {np.std(f1_scores):.4f})")

## c. Hybrid Fine-Tuning (Full Gold Standard)

In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoTokenizer, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset
from safetensors.torch import load_file

df_gold = pd.read_csv('/content/gold_standard_bio_labeled.csv')
df_gold = df_gold.dropna(subset=['Token', 'Label'])

sentences_all, labels_all = [], []
for seq_id, group in df_gold.groupby('Sentence_ID', sort=False):
    sentences_all.append(group['Token'].astype(str).tolist())
    labels_all.append(group['Label'].astype(str).tolist())

In [ ]:
base_model_path = "/content/indobertweet-crf-base-final"
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
label_list = ["O", "B-CRYPTO", "I-CRYPTO"]
label_to_id = {l: i for i, l in enumerate(label_list)}

def tokenize_and_align_labels(texts, tags):
    tokenized_inputs = tokenizer(texts, is_split_into_words=True, truncation=True, max_length=128, padding="max_length")
    aligned_labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None: label_ids.append(-100)
            elif word_idx != previous_word_idx:
                clean_label = label[word_idx] if label[word_idx] in label_to_id else "O"
                label_ids.append(label_to_id[clean_label])
            else: label_ids.append(-100)
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)
    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

train_encodings = tokenize_and_align_labels(sentences_all, labels_all)

train_dataset = Dataset.from_dict({'input_ids': train_encodings['input_ids'], 'attention_mask': train_encodings['attention_mask'], 'labels': train_encodings['labels']})

model = IndoBERTweetCRF("indolem/indobertweet-base-uncased", len(label_list))
state_dict = load_file(f"{base_model_path}/model.safetensors") if os.path.exists(f"{base_model_path}/model.safetensors") else torch.load(f"{base_model_path}/pytorch_model.bin", map_location='cpu')
model.load_state_dict(state_dict)
model.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class CRFTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]


final_training_args = TrainingArguments(
    output_dir="/content/temp_final",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2, # From K-Fold report
    weight_decay=0.01,
    save_strategy="no",
    logging_steps=50,
    fp16=True if torch.cuda.is_available() else False
)

trainer = CRFTrainer(
    model=model,
    args=final_training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer)
)

In [ ]:
trainer.train()

In [ ]:
final_model_path = "/content/drive/MyDrive/indobertweet-crf-PRODUCTION-FINAL"
os.makedirs(final_model_path, exist_ok=True)

torch.save(model.state_dict(), f"{final_model_path}/pytorch_model.bin")
tokenizer.save_pretrained(final_model_path)

## d. Error Analysis

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd

import collections
import collections.abc
collections.Iterable = collections.abc.Iterable

from transformers import AutoModel, AutoConfig, AutoTokenizer
from torchcrf import CRF
from safetensors.torch import load_file

df_gold = pd.read_csv('/content/gold_standard_bio_labeled.csv')
df_gold = df_gold.dropna(subset=['Token', 'Label'])

sentences_test = []
labels_test = []
for seq_id, group in df_gold.groupby('Sentence_ID', sort=False):
    sentences_test.append(group['Token'].astype(str).tolist())
    labels_test.append(group['Label'].astype(str).tolist())

print(f"[✔] Berhasil memuat {len(sentences_test)} kalimat.")

In [ ]:
class IndoBERTweetCRF(nn.Module):
    def __init__(self, model_checkpoint, num_labels):
        super(IndoBERTweetCRF, self).__init__()
        self.config = AutoConfig.from_pretrained(model_checkpoint)
        self.bert = AutoModel.from_pretrained(model_checkpoint, config=self.config)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)
        emissions = self.classifier(sequence_output)

        if labels is not None:
            pass
        else:
            crf_mask = attention_mask.type(torch.uint8)
            preds = self.crf.decode(emissions, mask=crf_mask)
            return {"logits": preds}

label_list = ["O", "B-CRYPTO", "I-CRYPTO"]

model_path = "/content/drive/MyDrive/indobertweet-crf-PRODUCTION-FINAL"

if not os.path.exists(model_path):
    raise FileNotFoundError(f"[{model_path} not found")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = IndoBERTweetCRF("indolem/indobertweet-base-uncased", len(label_list))

file_safetensors = os.path.join(model_path, "model.safetensors")
file_bin = os.path.join(model_path, "pytorch_model.bin")

if os.path.exists(file_safetensors):
    state_dict = load_file(file_safetensors)
elif os.path.exists(file_bin):
    state_dict = torch.load(file_bin, map_location=torch.device('cpu'))
else:
    raise FileNotFoundError("Model weight not dound")

model.load_state_dict(state_dict)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

In [ ]:
error_report = []
invalid_transitions = 0
total_tokens_predicted = 0

for i in range(len(sentences_test)):
    words = sentences_test[i]
    tags = labels_test[i]

    inputs_hf = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
    word_ids = inputs_hf.word_ids(batch_index=0)
    inputs = {k: v.to(device) for k, v in inputs_hf.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        preds = outputs["logits"][0]

    pred_labels_mapped = []
    previous_word_idx = None

    for j, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            if j < len(preds):
                pred_labels_mapped.append(label_list[preds[j]])
        previous_word_idx = word_idx

    for k in range(len(pred_labels_mapped)):
        total_tokens_predicted += 1
        current_label = pred_labels_mapped[k]

        if current_label == 'I-CRYPTO':
            if k == 0:
                invalid_transitions += 1
            else:
                previous_label = pred_labels_mapped[k-1]
                if previous_label not in ['B-CRYPTO', 'I-CRYPTO']:
                    invalid_transitions += 1

    is_error = False
    for true_lbl, pred_lbl in zip(tags, pred_labels_mapped):
        if true_lbl != pred_lbl:
            is_error = True
            break

    if is_error:
        kalimat_utuh = " ".join(words)
        detail_token = []
        for w, t, p in zip(words, tags, pred_labels_mapped):
            if t != p:
                detail_token.append(f"{w} (Actual: {t} | Prediciton: {p})")

        error_report.append({
            "Sentence": kalimat_utuh,
            "Error Detail": " || ".join(detail_token)
        })

In [ ]:
df_errors = pd.DataFrame(error_report)
export_path = "/content/error_report.csv"
df_errors.to_csv(export_path, index=False)

print(f"  - CRF Validation: {invalid_transitions} violation from total {total_tokens_predicted} token.")
print(f"  - Error Analysis: Found {len(df_errors)} sentencce with wrong prediction.")